# ComplicaSense · exploración y modelado

**Proyecto final de analítica de datos**

En este notebook se documenta el flujo completo: lectura del Excel, revisión de calidad, limpieza, creación de variables, entrenamiento de una regresión logística y una red neuronal artificial, evaluación y exportación de los modelos que consume la aplicación web.

> **Objetivo:** estimar la probabilidad de pertenecer al grupo `dengue complicado`. El resultado es académico y no constituye un diagnóstico médico.

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# El notebook funciona desde la raíz o desde la carpeta notebooks.
ROOT = Path.cwd()
if not (ROOT / 'MASTER_CHART_F1000Research.xlsx').exists():
    ROOT = ROOT.parent
DATASET = ROOT / 'MASTER_CHART_F1000Research.xlsx'
ARTIFACTS = ROOT / 'artifacts'
print(f'Carpeta de trabajo: {ROOT}')

## 1. Carga y exploración inicial

La variable objetivo está codificada como `1 = dengue no complicado` y `2 = dengue complicado`, según el README entregado con el dataset.

In [ ]:
raw = pd.read_excel(DATASET)
print(f'Registros: {raw.shape[0]} | Columnas: {raw.shape[1]}')
display(raw.head())
display(pd.DataFrame({
    'tipo': raw.dtypes.astype(str),
    'nulos': raw.isna().sum(),
    'unicos': raw.nunique(dropna=False)
}))

In [ ]:
# Distribución del objetivo antes de transformar códigos.
target_counts = raw.iloc[:, 0].value_counts().sort_index()
display(target_counts.rename(index={1: 'No complicado', 2: 'Complicado'}).to_frame('pacientes'))
print('Columnas constantes:', [c for c in raw.columns if raw[c].nunique(dropna=False) <= 1])

## 2. Limpieza y creación de variables

Se reutiliza `data_pipeline.py` para que el notebook y la aplicación web apliquen exactamente la misma transformación.

- Se eliminan columnas constantes (el dataset no tiene ninguna).
- Los códigos binarios `1/2` se convierten a `1/0`.
- `ALT`, `CREATININE` y `FERRITIN` se mantienen como valores faltantes para que el imputador aprenda la mediana solo con entrenamiento.
- Se crea `ferritin_missing` para conservar la señal de ausencia.
- Se agregan la razón plaquetas/WBC, la razón HB/PCV, el conteo de síntomas y el conteo de tratamientos.

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from data_pipeline import MODEL_FEATURES, prepare_training_frame

X, y, constant_columns = prepare_training_frame(raw)
print('Columnas constantes eliminadas:', constant_columns or 'Ninguna')
print(f'Matriz final: {X.shape[0]} filas x {X.shape[1]} variables')
display(X.head())
display(X.isna().sum().sort_values(ascending=False).head(10).to_frame('nulos'))

In [ ]:
display(pd.DataFrame({
    'grupo': ['Dengue no complicado', 'Dengue complicado'],
    'codigo_modelo': [0, 1],
    'pacientes': [(y == 0).sum(), (y == 1).sum()]
}))
display(X.describe().T[['mean', 'min', 'max']].round(2))

## 3. División y entrenamiento

La división estratificada reserva el 20% para prueba. Ambos modelos usan el mismo imputador y `StandardScaler`, evitando que las variables de laboratorio con escalas grandes dominen el aprendizaje.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

def make_pipeline(classifier):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', classifier),
    ])

logistic = make_pipeline(LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
neural = make_pipeline(MLPClassifier(
    hidden_layer_sizes=(24, 12), activation='relu', solver='adam',
    alpha=0.001, learning_rate_init=0.003, max_iter=1500,
    early_stopping=True, validation_fraction=0.2, n_iter_no_change=35,
    random_state=42
))

logistic.fit(X_train, y_train)
neural.fit(X_train, y_train)
print('Entrenamiento finalizado.')
print('Iteraciones de la red:', neural.named_steps['classifier'].n_iter_)

In [ ]:
def evaluate(model, name):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    return {
        'modelo': name,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, prob),
        'matriz_confusion': confusion_matrix(y_test, pred).tolist(),
    }

results = pd.DataFrame([evaluate(logistic, 'Regresión logística'), evaluate(neural, 'Red neuronal')])
display(results.style.format({c: '{:.3f}' for c in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']}))

### Interpretación rápida

- **Accuracy:** proporción total de aciertos.
- **Recall:** capacidad de encontrar casos complicados; es especialmente importante en un problema de riesgo.
- **ROC-AUC:** capacidad de ordenar casos de menor a mayor riesgo.
- La aplicación no muestra una decisión médica: presenta las probabilidades de ambos modelos para que sean comparadas.

In [ ]:
# Variables con mayor peso absoluto en la regresión logística.
coefficients = pd.Series(
    logistic.named_steps['classifier'].coef_[0], index=MODEL_FEATURES
).sort_values(key=np.abs, ascending=False)
display(coefficients.head(10).to_frame('coeficiente'))
print('Coeficiente positivo: aumenta la probabilidad estimada del grupo complicado.')

## 4. Exportación para la aplicación web

La aplicación Flask no vuelve a entrenar: carga este bundle de modelos y usa la misma función de ingeniería de variables. Ejecutar esta celda deja el despliegue listo.

In [ ]:
ARTIFACTS.mkdir(exist_ok=True)
def metric_dict(model):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    matrix = confusion_matrix(y_test, pred, labels=[0, 1])
    return {
        'accuracy': round(float(accuracy_score(y_test, pred)), 4),
        'precision': round(float(precision_score(y_test, pred, zero_division=0)), 4),
        'recall': round(float(recall_score(y_test, pred, zero_division=0)), 4),
        'f1': round(float(f1_score(y_test, pred, zero_division=0)), 4),
        'roc_auc': round(float(roc_auc_score(y_test, prob)), 4),
        'confusion_matrix': matrix.tolist(),
    }

metadata = {}
for column in ['hb', 'pcv', 'lowest_wbc', 'highest_wbc', 'platelet_count', 'alt', 'creatinine', 'ferritin']:
    values = X[column].dropna()
    metadata[column] = {'min': float(values.min()), 'max': float(values.max()), 'median': float(values.median())}

bundle = {
    'logistic': logistic, 'neural_network': neural,
    'feature_names': MODEL_FEATURES,
    'metrics': {'logistic': metric_dict(logistic), 'neural_network': metric_dict(neural)},
    'metadata': metadata,
    'dataset': {'rows': len(raw), 'raw_columns': raw.shape[1],
                'constant_columns_removed': constant_columns,
                'target_distribution': {'uncomplicated': int((y == 0).sum()), 'complicated': int((y == 1).sum())}},
}
joblib.dump(bundle, ARTIFACTS / 'models.joblib')
print(f"Bundle exportado en: {ARTIFACTS / 'models.joblib'}")

## 5. Próximo paso

Para probar la interfaz localmente, volver a la raíz del proyecto y ejecutar `python app.py`. Para Render, el `render.yaml` ejecuta `python train.py` durante el build y después inicia `gunicorn app:app`.